# Traffic Sign Recognition Project

## Setup
Import necessary libraries and set up paths.

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2

# Paths
data_path = 'data/processed/'
model_save_path = 'models/traffic_sign_model.h5'


## Data Preprocessing
### Load and Preprocess Data

In [ ]:

def load_images_from_folder(folder_path):
    images = []
    labels = []
    class_names = os.listdir(folder_path)
    for label, class_name in enumerate(class_names):
        class_folder = os.path.join(folder_path, class_name)
        if os.path.isdir(class_folder):
            for file_name in os.listdir(class_folder):
                file_path = os.path.join(class_folder, file_name)
                img = cv2.imread(file_path)
                if img is not None:
                    img = cv2.resize(img, (64, 64))
                    images.append(img)
                    labels.append(label)
    return np.array(images), np.array(labels), class_names

# Load Data
X_train, y_train, class_names = load_images_from_folder(os.path.join(data_path, 'train'))
X_val, y_val, _ = load_images_from_folder(os.path.join(data_path, 'val'))

# Normalize Data
X_train = X_train / 255.0
X_val = X_val / 255.0


### Data Augmentation

In [ ]:

data_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=False,
    fill_mode='nearest'
)

# Generate augmented data
train_gen = data_gen.flow(X_train, y_train, batch_size=32)


## Model Training
### Build and Train the Model

In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Build Model
model = build_cnn(input_shape=(64, 64, 3), num_classes=len(class_names))

# Define Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint(model_save_path, save_best_only=True, monitor='val_loss')

# Train Model
history = model.fit(
    train_gen,
    validation_data=(X_val, y_val),
    epochs=20,
    callbacks=[early_stopping, model_checkpoint]
)


### Evaluate Model

In [ ]:

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Training History')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


## Save and Load Model
### Save the Model

In [ ]:
model.save(model_save_path)

### Load the Model for Prediction

In [ ]:
model = load_model(model_save_path)

## Real-Time Prediction
### Predict on New Data

In [ ]:

def predict_new_images(image_paths, model, class_names):
    images = []
    for path in image_paths:
        img = cv2.imread(path)
        img = cv2.resize(img, (64, 64))
        img = img / 255.0
        images.append(img)

    images = np.array(images)
    predictions = model.predict(images)
    predicted_classes = [class_names[np.argmax(pred)] for pred in predictions]

    return predicted_classes

# Example Usage
new_image_paths = ['path_to_image1.jpg', 'path_to_image2.jpg']
predictions = predict_new_images(new_image_paths, model, class_names)

# Display Predictions
for path, pred in zip(new_image_paths, predictions):
    print(f"Image: {path} -> Predicted Class: {pred}")


### Visualization of Predictions

In [ ]:

plt.figure(figsize=(15, 5))
for i, path in enumerate(new_image_paths):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, len(new_image_paths), i + 1)
    plt.imshow(img)
    plt.title(predictions[i])
    plt.axis('off')
plt.show()


## API Integration
### Test API Endpoint

In [ ]:

import requests

url = "http://127.0.0.1:8000/predict/"
files = {'file': open('path_to_image.jpg', 'rb')}
response = requests.post(url, files=files)
print(response.json())
